In [0]:
# =============================================================================
# jobs/12_ea_area_boundaries_download.py
#
# Notebook:   12_ea_area_boundaries_download.py
# Pipeline:   FGS / EA Reference Data
# Author:     Jon Payne, Environment Agency
# Cadence:    ONE-OFF -- run manually from the Databricks workspace as needed.
#             This notebook is NOT called by any master polling notebook.
#             Re-run if the EA administrative boundaries change.
#
# PURPOSE
# -------
# Downloads the EA and Natural England public-face administrative area
# boundaries from the EA geoservices OGC API Features endpoint and writes
# them to a Delta table in Unity Catalog.
#
# Source:
#   https://environment.data.gov.uk/geoservices/datasets/
#   91d0fb43-209c-477f-91e3-74e756296268/ogc/features/v1/collections/
#   Administrative_Boundaries_Environment_Agency_and_Natural_England_
#   Public_Face_Areas/items
#
# The source is an OGC API Features endpoint returning GeoJSON. It is
# expected to contain 14 features (one per EA/NE administrative area).
# If the response contains more or fewer than 14, the notebook logs a
# warning but continues.
#
# CONFIRMED PROPERTY FIELDS (from API inspection 2026-05-29)
# -----------------------------------------------------------
#   identifier      -- integer ID
#   long_name       -- full area name, e.g. "Wessex"
#   short_name      -- abbreviated name, e.g. "Wessex"
#   code            -- area code, e.g. "WSX"
#   description     -- full text description of the area
#   seaward         -- "Yes" or "No"
#   date_from       -- ISO 8601 timestamp, area effective from
#   date_to         -- ISO 8601 timestamp, area effective to.
#                      A value of "1899-12-30..." is an ESRI null sentinel
#                      meaning open-ended -- stored as NULL in the table.
#   shape_leng      -- perimeter length in metres (float as string)
#   gdb_geomattr_data -- internal ESRI/GDB artefact, EXCLUDED from table
#
# OUTPUT TABLE
# ------------
# ea_administrative_areas
#
# One row per administrative area. Columns match the property fields above
# (minus gdb_geomattr_data), plus:
#   geometry_wkt  STRING    -- polygon geometry as WKT for spatial operations
#   ingested_at   TIMESTAMP -- when this notebook wrote the row
#
# The table is OVERWRITTEN on each run. It is a reference snapshot.
#
# MAP OUTPUT
# ----------
# After writing to Delta, the notebook renders an interactive Folium map
# showing all 14 areas as distinctly coloured polygons, with hover tooltips
# showing long_name and code. For visual verification only.
#
# DEPENDENCIES
# ------------
# - folium must be available on the cluster (pip install folium if not)
# - No authentication required -- OGC endpoint is open under OGL
# - prd_dash_lab.flood_forecasting_unrestricted schema must exist
# =============================================================================
#
# =============================================================================
# IMPORTS
# =============================================================================

import requests                           # HTTP request to OGC API
from datetime import datetime, timezone   # Ingestion timestamp
import pandas as pd                       # Build flat table
from shapely.geometry import shape        # Convert GeoJSON geometry to Shapely
from shapely import wkt as shapely_wkt    # WKT serialisation
import geopandas as gpd                   # Spatial dataframe for map rendering
import matplotlib.pyplot as plt           # Static map rendering
import matplotlib.patches as mpatches     # Legend patches
from pyspark.sql import SparkSession      # Write to Delta

# =============================================================================
# CONFIGURATION
# =============================================================================

CATALOG = "prd_dash_lab"
SCHEMA  = "flood_forecasting_unrestricted"
TABLE   = "ea_administrative_areas"
FULL_TABLE_NAME = f"{CATALOG}.{SCHEMA}.{TABLE}"

# OGC API Features endpoint.
# limit=50 is sufficient -- we expect 14 features.
# f=application/geo+json requests GeoJSON format explicitly.
API_URL = (
    "https://environment.data.gov.uk/geoservices/datasets/"
    "91d0fb43-209c-477f-91e3-74e756296268/ogc/features/v1/collections/"
    "Administrative_Boundaries_Environment_Agency_and_Natural_England_"
    "Public_Face_Areas/items"
    "?f=application%2Fgeo%2Bjson&limit=50"
)

EXPECTED_FEATURE_COUNT = 14
REQUEST_TIMEOUT        = 60    # This endpoint can be slow

# ESRI null sentinel for date_to -- means "open-ended, no end date".
# Any date_to starting with this string is stored as NULL.
ESRI_NULL_DATE = "1899-12-30"

# 14 distinct colours for the map -- one per area.
# Fixed list rather than a colourmap to ensure visual separation.
COLOURS = [
    "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd",
    "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf",
    "#aec7e8", "#ffbb78", "#98df8a", "#ff9896"
]


In [0]:
# =============================================================================
# STEP 1: FETCH FROM OGC API
# =============================================================================

print("=" * 60)
print("STEP 1: Fetch EA administrative area boundaries")
print("=" * 60)

ingested_at = datetime.now(timezone.utc)

try:
    response = requests.get(API_URL, timeout=REQUEST_TIMEOUT)
    response.raise_for_status()
except requests.exceptions.Timeout:
    raise Exception(
        f"OGC API timed out after {REQUEST_TIMEOUT}s. "
        "Try increasing REQUEST_TIMEOUT -- this endpoint is occasionally slow."
    )
except requests.exceptions.RequestException as e:
    raise Exception(f"OGC API request failed: {e}")

geojson        = response.json()
features       = geojson.get("features", [])
number_matched = geojson.get("numberMatched", "unknown")

print(f"  numberMatched (total in dataset): {number_matched}")
print(f"  Features returned:                {len(features)}")

if len(features) == 0:
    raise Exception("API returned zero features. Check the endpoint URL is still valid.")

if len(features) < EXPECTED_FEATURE_COUNT:
    print(f"  WARNING: Expected {EXPECTED_FEATURE_COUNT} features, got {len(features)}. "
          "The dataset may have changed or the API returned a partial response.")

if len(features) > EXPECTED_FEATURE_COUNT:
    print(f"  WARNING: Expected {EXPECTED_FEATURE_COUNT} features, got {len(features)}. "
          "More areas than anticipated -- check the source dataset.")


In [0]:
# =============================================================================
# STEP 2: PARSE FEATURES INTO A FLAT TABLE
# =============================================================================

print("=" * 60)
print("STEP 2: Parse GeoJSON features")
print("=" * 60)

rows = []

for i, feature in enumerate(features):
    props    = feature.get("properties", {})
    geom_raw = feature.get("geometry")

    # Convert GeoJSON geometry to WKT.
    # WKT (Well-Known Text) is a standard string representation of geometry
    # that GeoPandas, Shapely, PostGIS, and Spark spatial functions all read.
    if geom_raw:
        try:
            geometry_wkt = shape(geom_raw).wkt
        except Exception as e:
            print(f"  WARNING: Feature {i} geometry parse failed: {e}")
            geometry_wkt = None
    else:
        print(f"  WARNING: Feature {i} has no geometry.")
        geometry_wkt = None

    # Handle the date_to ESRI null sentinel.
    # "1899-12-30T00:00:00Z" means the area has no end date (still active).
    # Storing it as NULL is cleaner than a spurious 19th-century date.
    date_to_raw = props.get("date_to") or ""
    date_to     = None if date_to_raw.startswith(ESRI_NULL_DATE) else date_to_raw or None

    row = {
        "identifier":   str(props["identifier"]) if props.get("identifier") is not None else None,
        "long_name":    props.get("long_name"),
        "short_name":   props.get("short_name"),
        "code":         props.get("code"),
        "description":  props.get("description"),
        "seaward":      props.get("seaward"),       # "Yes" or "No"
        "date_from":    props.get("date_from"),     # ISO string
        "date_to":      date_to,                    # ISO string or None if open-ended
        "shape_leng":   str(props["shape_leng"]) if props.get("shape_leng") is not None else None,
        # gdb_geomattr_data intentionally excluded -- ESRI internal artefact
        "geometry_wkt": geometry_wkt,
        "ingested_at":  ingested_at,
    }

    rows.append(row)
    print(f"  {i+1:>2}. {str(row['long_name']):<25}  "
          f"code: {str(row['code']):<5}  "
          f"seaward: {str(row['seaward']):<3}  "
          f"geometry: {'ok' if geometry_wkt else 'MISSING'}")

print(f"\n  Parsed {len(rows)} rows.")


In [0]:
# =============================================================================
# STEP 3: WRITE TO DELTA TABLE
# =============================================================================

print("=" * 60)
print("STEP 3: Write to Delta table")
print("=" * 60)

spark = SparkSession.builder.getOrCreate()

df_pd    = pd.DataFrame(rows)
df_spark = spark.createDataFrame(df_pd)

# OVERWRITE -- this is a reference snapshot, not a log.
# overwriteSchema: true allows the schema to evolve if the API adds fields
# in a future version of the dataset.
(df_spark.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(FULL_TABLE_NAME))

print(f"  Written {len(rows)} rows to {FULL_TABLE_NAME}")
print(f"  Columns: {list(df_pd.columns)}")


In [0]:
# =============================================================================
# STEP 4: VERIFY -- READ BACK FROM DELTA
# =============================================================================

print("=" * 60)
print("STEP 4: Verify -- read back from Delta")
print("=" * 60)

verify_df = spark.table(FULL_TABLE_NAME)
print(f"  Row count in table: {verify_df.count()}")
display(verify_df.drop("geometry_wkt"))   # Suppress WKT for readability


In [0]:
# =============================================================================
# STEP 5: MAP VISUALISATION
# =============================================================================
# Static matplotlib map of all EA administrative areas.
# Each area is a distinctly coloured polygon labelled at its centroid.
# Renders as a PNG directly in the notebook cell -- no output size limit.
# For visual verification only -- confirm geometries before downstream use.
# =============================================================================

print("=" * 60)
print("STEP 5: Render map")
print("=" * 60)

# Build a GeoDataFrame from the parsed rows.
# We re-parse from the stored WKT strings to confirm the round-trip
# (GeoJSON -> Shapely -> WKT -> Shapely) is clean.
gdf_rows = []
for row in rows:
    if row["geometry_wkt"]:
        try:
            geom = shapely_wkt.loads(row["geometry_wkt"])
            gdf_rows.append({**row, "geometry": geom})
        except Exception as e:
            print(f"  WARNING: Could not re-parse WKT for {row.get('long_name')}: {e}")

gdf = gpd.GeoDataFrame(gdf_rows, geometry="geometry", crs="EPSG:4326")
# Filter to non-tidal boundaries for the plot.
# All 24 rows (including tidal variants) are stored in Delta -- this is
# display only. Use seaward = 'No' to get the 14 standard boundaries.
gdf = gdf[gdf["seaward"] == "No"].copy()

print(f"  GeoDataFrame: {len(gdf)} rows with valid geometry.")

# matplotlib renders a static PNG directly into the notebook cell.
# No HTML, no JavaScript, no output size limit.
# Full-precision geometry is used -- no simplification needed for a static plot.

fig, ax = plt.subplots(figsize=(10, 12))

for idx, (_, row) in enumerate(gdf.iterrows()):
    colour    = COLOURS[idx % len(COLOURS)]
    long_name = str(row.get("long_name", f"Area {idx+1}"))

    gdf.iloc[[idx]].plot(
        ax           = ax,
        color        = colour,
        edgecolor    = "white",
        linewidth    = 0.8,
        alpha        = 0.7,
    )

    # Label each area at its centroid
    centroid = row["geometry"].centroid
    ax.annotate(
        long_name,
        xy         = (centroid.x, centroid.y),
        fontsize   = 6,
        ha         = "center",
        va         = "center",
        color      = "black",
        fontweight = "bold",
    )

ax.set_title("EA Administrative Areas", fontsize=14, pad=12)
ax.set_axis_off()
plt.tight_layout()
plt.show()
print("  Map rendered.")


In [0]:
# =============================================================================
# DONE
# =============================================================================

print("=" * 60)
print("Job 12 complete.")
print(f"  Table:  {FULL_TABLE_NAME}")
print(f"  Rows:   {len(rows)}")
print(f"  Map:    rendered in cell above -- verify before downstream use")
print("=" * 60)
